# Raster data - the basics

This tutorial focuses on working with basic Raster data management and analysis using `plans`.

## Notebook setup

For users running this tutorial as a Jupyter Notebook, this cell must be executed first:

In [ ]:
import sys
from pathlib import Path
import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Install `plans` in `google.colab`.
# Use `pip install plans` for other environments.

if "google.colab" in sys.modules:
    import os
    os.system(f"{sys.executable} -m pip install -q plans")

# This avoids warnings related to uninstalled fonts
import logging
# Set the matplotlib font manager logger to only show errors (hides warnings)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# define output folder
OUTPUT_DIR = Path("outputs/raster")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: ./{OUTPUT_DIR}")

## The `Raster` object

The `Raster` object is a primitive class that lives under `plans.datasets` module. The `Raster` object stores all core methods for working with single-banded, gridded spatial data (maps).

Import `Raster` object:

In [ ]:
from plans.datasets import Raster

Create an instance of the `Raster`:

In [ ]:
rs = Raster(name="Testing", alias="tst")

Check out the `rs` variable type:

In [ ]:
print(type(rs))

Edit and inspect attributes:

In [ ]:
rs.units = "cm"
rs.description = "Just a tutorial"
print(rs)

## Working with data

Raster data files are expected to be in the standard `tif` (GeoTiff) format.

For the `Raster` core object, data type is parsed as `float32`, that is, the data is treated as real numbers by default (instead of integers, for example).


### Setup a built-in example project

Access simple example raster files via the `TOY_PROJECT` feature:

In [ ]:
from plans.config import TOY_PROJECT
from plans.project import load_project
pj = load_project(TOY_PROJECT)

### Load data from tif file

Let's now get the slope map from the `topo` folder:

In [ ]:
raster_file = Path(pj.folder_topo) / "slope.tif"

Then use `load_data()`

In [ ]:
rs.load_data(file_data=raster_file)

### Inspect and view data

Inspect raster metadata:

In [ ]:
pprint.pp(rs.raster_metadata)

Inspect data (numpy array):

In [ ]:
print(rs.data)

Get basic statistics in a `dict`:

In [ ]:
stats_dict = rs.get_stats()
pprint.pp(stats_dict)

View map:

In [ ]:
rs.view()

Convert and get an `Univar` instance:

In [ ]:
uv = rs.get_univar()
print(type(uv))

View as Univar:

In [ ]:
uv.view()

### Export to other file

Tif format is exported by default.

In [ ]:
fo = rs.export(folder=OUTPUT_DIR, filename="slope_example")
print(fo)
print(f"File was created: {Path(fo).is_file()}")

### Copy raster structure

Sometimes it is useful to process raster data and then save it back to the same raster grid.

Example: making a brand new data grid with the same size using `numpy.random`

In [ ]:
new_data = np.random.normal(loc=100, scale=10, size=np.shape(rs.data))
print(type(new_data))

Define new raster, copy structure from other and set data:

In [ ]:
# define a new raster
rs_new = Raster()
# copy structure from rs object
rs_new.copy_structure(raster_ref=rs)
# set data
rs_new.set_data(grid=new_data)

View and/or save

In [ ]:
rs_new.view()

In [ ]:
rs_new.export(folder=OUTPUT_DIR, filename="new_data")

## Types of Rasters

The `Raster` class is extented into several families or types.

### `SciRaster` family

This is a large family of rasters that are numerical values. Behaviour is nearly the same from `Raster` base class, but `NODATA_value` and `dtype` is enforced to the incoming data.

Members of this family are all conventional quantitative spatial variables, like terrain elevation or soil water content.

In [ ]:
from plans.datasets import SciRaster

In [ ]:
sciraster = SciRaster()
sciraster.load_data(file_data=raster_file)
print(f"Data type: {sciraster.dtype}")
print(f"No-data value: {sciraster.raster_metadata['NODATA_value']}")

View layout is the same from `Raster`:

In [ ]:
sciraster.view()

### `QualiRaster` family

This is a large family of rasters that supports categorical maps, eg., land use maps. Data is always accompanied by an auxiliar Attribute Table with `id`, `name`, `alias` and `color` fields:

In [ ]:
from plans.datasets import QualiRaster

Define new files (raster and table):

In [ ]:
quali_raster_file = Path(pj.folder_lulc) / "observed/lulc_2020-01-01.tif"
quali_raster_table = Path(pj.folder_lulc) / "observed/lulc_attributes.csv"

In [ ]:
qualiraster = QualiRaster()
qualiraster.load_data(
    file_data=quali_raster_file,
    file_table=quali_raster_table
)
print(f"Data type: {qualiraster.dtype}")
print(f"No-data value: {qualiraster.raster_metadata['NODATA_value']}")

Access the Attribute Table:

In [ ]:
qualiraster.table.head()

Get areas table:

In [ ]:
areas_df = qualiraster.get_areas()
areas_df

View scheme provides a horizontal bar for aereal ratios:

In [ ]:
qualiraster.view()

### Dedicated extensions

`plans.datasets.spatial` offers a series of dedicated `Rasters` for spatial variables. This helps to shortcut attributes and visual setup. Some examples are provided below.

#### `Elevation` raster

In [ ]:
from plans.datasets.spatial import Elevation
dem_file = Path(pj.folder_topo) / "dem.tif"
dem_raster = Elevation()
dem_raster.load_data(file_data=dem_file)
dem_raster.view_specs["range"] = [800, 1200]
dem_raster.view()

#### `Slope` raster

In [ ]:
from plans.datasets.spatial import Slope
slope_file = Path(pj.folder_topo) / "slope.tif"
slope_raster = Slope()
slope_raster.load_data(file_data=slope_file)
slope_raster.view()

#### `HAND` raster

In [ ]:
from plans.datasets.spatial import HAND
hand_file = Path(pj.folder_topo) / "hand.tif"
hand_raster = HAND()
hand_raster.load_data(file_data=hand_file)
hand_raster.view()

#### `TWI` raster

In [ ]:
from plans.datasets.spatial import TWI
twi_file = Path(pj.folder_topo) / "twi.tif"
twi_raster = TWI()
twi_raster.load_data(file_data=twi_file)
twi_raster.view()

#### `LDD` raster

LDD (Local Drain Direction) is a core map for hydrological analysis. It depends on a convention of integer numbers.

In [ ]:
from plans.datasets.spatial import LDD

ldd_file = Path(pj.folder_topo) / "ldd.tif"
ldd_raster = LDD()
ldd_raster.load_data(file_data=ldd_file)
ldd_raster.view()

#### `Soils` raster

Soils may reflect soils classes and lithology. Requires the Attribute Table.

In [ ]:
from plans.datasets.spatial import Soils
# define file
soils_file = Path(pj.folder_soils) / "soils.tif"
# define table
soils_table = Path(pj.folder_soils) / "soils_attributes.csv"
# define raster
soils_raster = Soils()
soils_raster.load_data(file_data=soils_file, file_table=soils_table)
soils_raster.view()

#### `LULC` raster

LULC (Land Use and Land Cover) requires the Attribute Table and date to be provided.

In [ ]:
from plans.datasets.spatial import LULC
# define file
lulc_file = Path(pj.folder_lulc) / "observed/lulc_2020-01-01.tif"
# define table
lulc_table = Path(pj.folder_lulc) / "observed/lulc_attributes.csv"
# define name and datetime
lulc_raster = LULC(name="LULC", datetime="2020-01-01")
lulc_raster.load_data(file_data=lulc_file, file_table=lulc_table)
lulc_raster.view_specs["title"] = "Land Use in 2020"
lulc_raster.view()


#### `AOI` Raster

The `AOI` Raster (Area-Of-Interest) is a member of the `QualiRaster` family.

It is a boolean grid used mostly for masking operations. In this case, there is no need for an Attribute Table to be provided.

In [ ]:
from plans.datasets import AOI
aoi = AOI()
# define basin raster for the AOI
aoi_file = Path(pj.folder_basins) / "main/basin.tif"
aoi.load_data(file_data=aoi_file)
aoi.view()

## Masking Operations

Masking is a very basic operation when working with Rasters, since in most cases we need to filter the data to a given AOI, like a catchment basin.

Say we want to know the average slope of a basin:

In [ ]:
# use the data from the aoi map
slope_raster.apply_aoi_mask(grid_aoi=aoi.data)
slope_raster.view_specs["title"] = "Slope map for the basin"
slope_raster.view()

The AOI mask can be realeased:

In [ ]:
slope_raster.release_aoi_mask()
slope_raster.view_specs["title"] = "Slope map for the full extension"
slope_raster.view()

The same can be done with qualitative maps:

In [ ]:
lulc_raster.apply_aoi_mask(grid_aoi=aoi.data)
lulc_raster.view_specs["title"] = "Land Use of the basin in 2020"
lulc_raster.view()